# 04 MobileNetV3 Training

Project: Cat Breed Classification Using CNN Transfer Learning  
Domain: Animal subspecies  
Model: MobileNetV3Small

This notebook trains and evaluates a MobileNetV3Small transfer learning model for the selected cat breed classification task.

Required previous step: run `01_data_preparation.ipynb` first so the dataset folders exist.

## 1. Setup

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import accuracy_score, average_precision_score, classification_report, confusion_matrix

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))

Optional for Google Colab: uncomment the Drive mount if your project folder is stored in Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = Path('/content/drive/MyDrive/AI-Image-Classification-Project')

PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset"
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

RESULTS_DIR = PROJECT_ROOT / "results"
GRAPHS_DIR = RESULTS_DIR / "graphs"
CONFUSION_DIR = RESULTS_DIR / "confusion_matrices"
MODELS_DIR = PROJECT_ROOT / "models"

for folder in [RESULTS_DIR, GRAPHS_DIR, CONFUSION_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset folder exists:", DATASET_DIR.exists())

## 2. Load Dataset

Images are loaded from folders created by the data preparation notebook.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 3. Build MobileNetV3 Transfer Learning Model

The MobileNetV3Small base uses ImageNet pretrained weights. The base model is frozen first, and a new classification head is trained for the 5 cat breeds.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="data_augmentation")

base_model = MobileNetV3Small(
    weights="imagenet",
    include_top=False,
    include_preprocessing=False,
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs, name="resnet50_cat_breed_classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 4. Train Model For 50 Epochs

The assessment asks for 50 epochs. Early stopping is included to protect against overfitting, but if your lecturer requires exactly 50 completed epochs, remove the `EarlyStopping` callback from the list below.

In [ ]:
EPOCHS = 50
MODEL_NAME = "mobilenetv3"
model_path = MODELS_DIR / f"{MODEL_NAME}_cat_breed.keras"

callbacks = [
    ModelCheckpoint(
        filepath=model_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=5,
        min_lr=1e-7,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    ),
]

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

training_time_seconds = time.time() - start_time
training_time_minutes = training_time_seconds / 60

print(f"Training time: {training_time_minutes:.2f} minutes")

## 5. Plot Accuracy And Loss

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_history.csv", index=False)
history_df.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("MobileNetV3 Training And Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / f"{MODEL_NAME}_accuracy.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("MobileNetV3 Training And Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / f"{MODEL_NAME}_loss.png", dpi=150)
plt.show()

## 6. Evaluate On Test Dataset

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

In [ ]:
y_true = []
y_prob = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_prob.append(predictions)
    y_true.append(labels.numpy())

y_true = np.vstack(y_true)
y_prob = np.vstack(y_prob)

y_true_labels = np.argmax(y_true, axis=1)
y_pred_labels = np.argmax(y_prob, axis=1)

test_accuracy_from_predictions = accuracy_score(y_true_labels, y_pred_labels)
mean_average_precision = average_precision_score(y_true, y_prob, average="macro")

print("Test accuracy from predictions:", test_accuracy_from_predictions)
print("Mean average precision:", mean_average_precision)

## 7. Confusion Matrix And Classification Report

In [ ]:
cm = confusion_matrix(y_true_labels, y_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title("MobileNetV3 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(CONFUSION_DIR / f"{MODEL_NAME}_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
report = classification_report(
    y_true_labels,
    y_pred_labels,
    target_names=class_names,
    output_dict=True,
)

report_df = pd.DataFrame(report).transpose()
report_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_classification_report.csv")
report_df

## 8. Save Model Metrics

These values will be used later in `05_model_comparison.ipynb`.

In [ ]:
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
non_trainable_params = int(np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))
total_params = trainable_params + non_trainable_params

metrics = {
    "model": "MobileNetV3Small",
    "dataset": "Oxford-IIIT Pet Dataset - 5 cat breeds",
    "classes": class_names,
    "epochs_requested": EPOCHS,
    "epochs_completed": len(history_df),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_accuracy_from_predictions": float(test_accuracy_from_predictions),
    "mean_average_precision_macro": float(mean_average_precision),
    "training_time_seconds": float(training_time_seconds),
    "training_time_minutes": float(training_time_minutes),
    "total_params": total_params,
    "trainable_params": trainable_params,
    "non_trainable_params": non_trainable_params,
}

with open(RESULTS_DIR / f"{MODEL_NAME}_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

metrics

## 9. Notes For Report

When writing the final report or preparing the presentation, mention:

- MobileNetV3Small used ImageNet pretrained weights.
- The base CNN layers were frozen during the first training stage.
- A new classification head was trained for the five selected cat breeds.
- The model was evaluated using test accuracy, mAP, training time, parameter count, and confusion matrix.

Next notebooks:

- `05_model_comparison.ipynb`